# DDPM
拡散モデル

## import

In [9]:
import torch
import torch.nn as nn


## Use GPU if Available

In [10]:
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print(device)


cpu


## U-Net
スキップ接続を利用して大域的な特徴と局所的な特徴の両方を捉える，畳み込みニューラルネットワーク．

In [11]:
class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.convs = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )
    
    def forward(self, x):
        return self.convs(x)

class UNet(nn.Module):
    def __init__(self, in_ch=1):
        super().__init__()

        self.down1 = ConvBlock(in_ch, 64)
        self.down2 = ConvBlock(64, 128)
        self.bot1 = ConvBlock(128, 256)
        self.up2 = ConvBlock(128 + 256, 128)
        self.up1 = ConvBlock(64 + 128, 64)
        self.out = nn.Conv2d(64, in_ch, kernel_size=1) # シンプルな特徴マップの重み付き和

        self.maxpool = nn.MaxPool2d(2)
        self.upsample = nn.Upsample(scale_factor=2, mode='bilinear')

    def forward(self, x):
        x1 = self.down1(x)
        x = self.maxpool(x1)
        x2 = self.down2(x)
        x = self.maxpool(x2)

        x = self.bot1(x)

        x = self.upsample(x)
        x = torch.cat([x, x2], dim=1) # チャネル次元で結合
        x = self.up2(x)
        x = self.upsample(x)
        x = torch.cat([x, x1], dim=1) # チェネル次元で結合
        x = self.up1(x)

        x = self.out(x)

        return x

model = UNet()
x = torch.randn(10, 1, 28, 28) # dummy input
y = model(x)
print(y.shape)


torch.Size([10, 1, 28, 28])
